<a href="https://colab.research.google.com/github/mohanad-yasser/hotel-booking/blob/work/draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

users_df   = pd.read_csv("dataset/users.csv")
hotels_df  = pd.read_csv("dataset/hotels.csv")
reviews_df = pd.read_csv("dataset/reviews.csv")

users_drop_cols   = ["join_date"]
hotels_drop_cols  = ["hotel_name", "star_rating", "lat", "lon"]
reviews_drop_cols = ["review_text","review_date"]


users_df.drop(columns=users_drop_cols, errors='ignore', inplace=True)
hotels_df.drop(columns=hotels_drop_cols, errors='ignore', inplace=True)
reviews_df.drop(columns=reviews_drop_cols, errors='ignore', inplace=True)


print(f" Users dataframe shape after drop  : {users_df.shape}")
print(f" Hotels dataframe shape after drop : {hotels_df.shape}")
print(f" Reviews dataframe shape after drop: {reviews_df.shape}")


display(users_df.head())
display(hotels_df.head())
display(reviews_df.head())
users_df.to_csv("dataset/helper/users_cleaned.csv", index=False)
hotels_df.to_csv("dataset/helper/hotels_cleaned.csv", index=False)
reviews_df.to_csv("dataset/helper/reviews_cleaned.csv", index=False)


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("dataset/helper")

users   = pd.read_csv(DATA_DIR / "users_cleaned.csv")
hotels  = pd.read_csv(DATA_DIR / "hotels_cleaned.csv")
reviews = pd.read_csv(DATA_DIR / "reviews_cleaned.csv")



users = users.rename(columns={"country": "user_country"})
hotels = hotels.rename(columns={"country": "hotel_country"})

merged_df = pd.merge(reviews, users, on="user_id", how="left")


merged_df = pd.merge(merged_df, hotels, on="hotel_id", how="left")

print(f"✅ Final merged dataset shape: {merged_df.shape}")
display(merged_df.head())


print("\nMissing values per column:")
print(merged_df.isnull().sum())

merged_df.to_csv("dataset/helper/hotel_reviews_merged.csv", index=False)
print("\n Merged dataset saved to: dataset/helper/hotel_reviews_merged.csv")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt, ceil

DATA_PATH = "dataset/helper/hotel_reviews_merged.csv"
df = pd.read_csv(DATA_PATH)



score_cols = [
    "score_overall","score_cleanliness","score_comfort","score_facilities",
    "score_location","score_staff","score_value_for_money"
]
base_cols = [
    "cleanliness_base","comfort_base","facilities_base",
    "location_base","staff_base","value_for_money_base"
]

df = df.dropna(subset=["traveller_type","city"])

for c in score_cols:
    if c in df.columns:
        df[c] = df[c].clip(0, 10)

df["composite"] = df[score_cols].mean(axis=1)


df["delta_mean"] = (
    pd.concat([
        df["score_cleanliness"]     - df["cleanliness_base"],
        df["score_comfort"]         - df["comfort_base"],
        df["score_facilities"]      - df["facilities_base"],
        df["score_location"]        - df["location_base"],
        df["score_staff"]           - df["staff_base"],
        df["score_value_for_money"] - df["value_for_money_base"],
    ], axis=1)
    .mean(axis=1)
)

def agg_block(g: pd.DataFrame) -> pd.Series:
    n   = g.shape[0]
    m   = g["composite"].mean()
    med = g["composite"].median()
    sd  = g["composite"].std(ddof=1)
    ci  = 1.96 * (sd / sqrt(n)) if n >= 30 and pd.notna(sd) else np.nan  # normal approx
    m_delta = g["delta_mean"].mean()
    return pd.Series({
        "n": n,
        "mean_composite": m,
        "median_composite": med,
        "std": sd,
        "ci95": ci,
        "mean_delta": m_delta
    })

agg = (
    df.groupby(["traveller_type","city"], as_index=False, sort=False)
      .apply(agg_block)
      .reset_index(drop=True)
)


type_counts = df.groupby("traveller_type").size().rename("total_n")
agg = agg.merge(type_counts, on="traveller_type", how="left")
agg["min_n_type"] = (agg["total_n"] * 0.01).apply(lambda x: max(30, int(ceil(x))))
agg = agg[agg["n"] >= agg["min_n_type"]].copy()

agg = agg.sort_values(
    ["traveller_type","mean_composite","n","median_composite","city"],
    ascending=[True, False, False, False, True]
)
agg["rank"] = agg.groupby("traveller_type").cumcount() + 1

winners = agg[agg["rank"] == 1].copy()
top3    = agg[agg["rank"] <= 3].copy()

display(winners.sort_values("traveller_type"))
display(top3.sort_values(["traveller_type","rank","city"]))


winners_path = "dataset/helper/out_best_city_per_traveller_type.csv"
top3_path    = "dataset/helper/out_top3_cities_per_traveller_type.csv"
agg_path     = "dataset/helper/out_all_city_stats.csv"

winners.to_csv(winners_path, index=False)
top3.to_csv(top3_path, index=False)
agg.drop(columns=["total_n","min_n_type"]).to_csv(agg_path, index=False)

print(f" Saved: {winners_path}")
print(f" Saved: {top3_path}")
print(f" Saved: {agg_path}")


top5 = (
    agg.sort_values(["traveller_type","mean_composite","n"], ascending=[True, False, False])
       .groupby("traveller_type")
       .head(5)
)

plot_paths = []
for ttype, sub in top5.groupby("traveller_type"):
    sub = sub.sort_values("mean_composite", ascending=True)
    y = np.arange(len(sub))

    plt.figure(figsize=(9, 5))
    bars = plt.barh(y, sub["mean_composite"])

    if sub["ci95"].notna().any():
        plt.errorbar(sub["mean_composite"], y, xerr=sub["ci95"].fillna(0), fmt='none', capsize=3)


    for i, v in enumerate(sub["mean_composite"]):
        plt.text(v, i, f" {v:.3f}", va="center")


    lo = max(0, sub["mean_composite"].min() - 0.1)
    hi = min(10, sub["mean_composite"].max() + 0.1)
    plt.xlim(lo, hi)

    plt.yticks(y, sub["city"])
    plt.xlabel("Mean Composite Score (0–10)")
    min_n = int(sub["min_n_type"].iloc[0])
    plt.title(f"Top-5 Cities for {ttype} (±95% CI, n≥{min_n})")
    plt.tight_layout()

    out_png = f"dataset/helper/plot_top5_{ttype.replace(' ','_').lower()}.png"
    plt.savefig(out_png, dpi=150)
    plot_paths.append(out_png)
    plt.show()

print(" Saved charts:")
for p in plot_paths:
    print("   -", p)


win_plot = winners.sort_values("traveller_type").copy()
plt.figure(figsize=(8, 4))
x = np.arange(win_plot.shape[0])
plt.bar(x, win_plot["mean_composite"])
for i, (city, val) in enumerate(zip(win_plot["city"], win_plot["mean_composite"])):
    plt.text(i, val, f"{city}\n{val:.3f}", ha="center", va="bottom")

plt.xticks(x, win_plot["traveller_type"], rotation=0)
plt.ylabel("Mean Composite Score (0–10)")
plt.title("Best City per Traveller Type")
plt.tight_layout()
out_png = "dataset/helper/plot_winners_summary.png"
plt.savefig(out_png, dpi=150)
print(" Saved chart:", out_png)
plt.show()


Best city for each traveller type.
For Business Amsterdam.
For Couple Dubai.
For Family Dubai.
For Solo Dubai.


In [ ]:

import pandas as pd
import numpy as np


DATA_PATH = "dataset/helper/hotel_reviews_merged.csv"
df = pd.read_csv(DATA_PATH)


NEEDED = ["age_group", "hotel_country", "score_value_for_money"]
missing = [c for c in NEEDED if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df.dropna(subset=NEEDED).copy()
df["score_value_for_money"] = df["score_value_for_money"].clip(0, 10)


agg = (
    df.groupby(["age_group", "hotel_country"], as_index=False)
      .agg(
          n=("score_value_for_money", "size"),
          median_vfm=("score_value_for_money", "median"),
      )
)


MIN_N = 30
agg = agg[agg["n"] >= MIN_N].copy()


agg = agg.sort_values(
    ["age_group", "median_vfm", "n", "hotel_country"],
    ascending=[True, False, False, True]
)
agg["rank"] = agg.groupby("age_group").cumcount() + 1


top3 = (
    agg[agg["rank"] <= 3][["age_group", "hotel_country", "n", "median_vfm"]]
      .sort_values(["age_group", "median_vfm", "n", "hotel_country"],
                   ascending=[True, False, False, True])
      .reset_index(drop=True)
)

print(" Top-3 countries by Median Value-for-Money per age group:")
display(top3)


top3.to_csv("dataset/helper/out_top3_vfm_countries_per_agegroup.csv", index=False)



COLS = ["age_group", "hotel_country", "n", "median_vfm"]

top3_slim = (
    top3[COLS]
    .sort_values(["age_group", "median_vfm", "n", "hotel_country"],
                 ascending=[True, False, False, True])
    .reset_index(drop=True)
)




top3_path = "dataset/helper/out_top3_vfm_countries_per_agegroup.csv"
top3_slim.to_csv(top3_path, index=False)
print(f" Saved: {top3_path}")


all_path = "dataset/helper/out_all_vfm_country_stats.csv"
agg.to_csv(all_path, index=False)
print(f" Saved: {all_path}")




for age, sub in top3_slim.groupby("age_group"):
    sub = sub.sort_values("median_vfm", ascending=True)
    y = np.arange(len(sub))

    plt.figure(figsize=(9, 5))
    plt.barh(y, sub["median_vfm"])
    plt.yticks(y, sub["hotel_country"])
    plt.xlabel("Median Value-for-Money (0–10)")
    plt.title(f"Top-3 Countries by Median Value-for-Money — Age Group: {age} (n≥{MIN_N})")

    for i, val in enumerate(sub["median_vfm"]):
        plt.text(val, i, f" {val:.3f}", va="center")

    lo = max(0, sub["median_vfm"].min() - 0.1)
    hi = min(10, sub["median_vfm"].max() + 0.1)
    plt.xlim(lo, hi)

    plt.tight_layout()
    plt.show()


Top 3 countries with the best value-for-money score per
traveler’s age group.



Age Group 18-24.
China, Netherlands and Canada.

Age Group 25-34. China, New Zealand and Spain.

Age Group 35-45. Netherlands, Canada and New Zealand.

Age Group 45-55. China, New Zealand and Thailand.

Age Group 55+. Netherlands, New Zealand and Spain.

In [ ]:
df = pd.read_csv("dataset/helper/hotel_reviews_merged.csv")
country_group_mapping = {
    "United States": "North_America",
    "Canada": "North_America",
    "Germany": "Western_Europe",
    "France": "Western_Europe",
    "United Kingdom": "Western_Europe",
    "Netherlands": "Western_Europe",
    "Spain": "Western_Europe",
    "Italy": "Western_Europe",
    "Russia": "Eastern_Europe",
    "China": "East_Asia",
    "Japan": "East_Asia",
    "South Korea": "East_Asia",
    "Thailand": "Southeast_Asia",
    "Singapore": "Southeast_Asia",
    "United Arab Emirates": "Middle_East",
    "Turkey": "Middle_East",
    "Egypt": "Africa",
    "Nigeria": "Africa",
    "South Africa": "Africa",
    "Australia": "Oceania",
    "New Zealand": "Oceania",
    "Brazil": "South_America",
    "Argentina": "South_America",
    "India": "South_Asia",
    "Mexico": "North_America_Mexico"
}


df["country_group"] = df["hotel_country"].map(country_group_mapping)
df.to_csv("dataset/helper/hotel_reviews_merged_country_groups.csv", index=False)

In [ ]:
df = pd.read_csv("dataset/helper/hotel_reviews_merged_country_groups.csv")


selected_features = [
    "country_group",
    "score_overall", "score_cleanliness", "score_comfort", "score_facilities", "score_location","user_country",
    "score_staff", "score_value_for_money", "user_gender", "age_group", "traveller_type",
   "cleanliness_base", "comfort_base", "facilities_base", "location_base",
    "staff_base", "value_for_money_base"
]


existing_features = [col for col in selected_features if col in df.columns]
numerical_features = ["score_overall", "score_cleanliness", "score_comfort", "score_facilities", "score_location", "score_staff", "score_value_for_money","cleanliness_base", "comfort_base", "facilities_base", "location_base",
    "staff_base", "value_for_money_base"]
categorical_features = ["user_gender", "age_group", "traveller_type"]

filtered_df = df[existing_features]


output_path = "dataset/helper/predict_country_groups.csv"
filtered_df.to_csv(output_path, index=False)

print(f" File saved successfully at: {output_path}")
print("Included columns:", existing_features)

In [ ]:
import pandas as pd


df = pd.read_csv("dataset/helper/predict_country_groups.csv")


categorical_features = ["age_group", "traveller_type","user_country"]


df['user_gender'] = df['user_gender'].map({'Male': 0, 'Female': 1})

df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)



df_encoded.to_csv("dataset/final_dataset.csv", index=False)

print(" One-hot encoding complete. Saved as final_dataset.csv")


Leakage


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("dataset/final_dataset.csv")


df = df.dropna(subset=["country_group"])


X = df.drop(columns=["country_group"], errors='ignore')
leakage_cols = [col for col in X.columns if "cleanliness_base" in col.lower() or "comfort_base" in col.lower() or "staff_base" in col.lower() or "facilities_base" in col.lower() or "value_for_money_base" in col.lower()]
X = X.drop(columns=leakage_cols)
print(X.columns)
y = df["country_group"]
le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train = X_train.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
X_test  = X_test.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)






model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)
model.fit(X_train, y_train)


importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values(by="importance", ascending=False)

print(importance_df)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(importance_df["feature"], importance_df["importance"], color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()


y_train_pred = model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f"🎯 Training Accuracy: {train_acc:.4f}")
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
plt.figure(figsize=(5, 5))
plt.bar(["Training", "Test"], [train_acc, test_acc], color=["skyblue", "lightcoral"])
plt.title("Training vs Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
print(" Accuracy:", accuracy_score(y_test, y_pred))
print("\n Classification Report:")
print(classification_report(y_test, y_pred))
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import pandas as pd


report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()


class_report_df = report_df.iloc[:-3, :]


plt.figure(figsize=(10, 6))
sns.barplot(data=class_report_df[['precision', 'recall', 'f1-score']])
plt.title("Classification Metrics per Class")
plt.xlabel("Metric")
plt.ylabel("Score (0–1)")
plt.ylim(0, 1)
plt.legend(labels=['Precision', 'Recall', 'F1-Score'])
plt.tight_layout()
plt.show()
print("\n Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


No leakage using one base


In [ ]:


df = pd.read_csv("dataset/final_dataset.csv")


df = df.dropna(subset=["country_group"])


X = df.drop(columns=["country_group"], errors='ignore')
leakage_cols = [col for col in X.columns if "cleanliness_base" in col.lower() or "comfort_base" in col.lower() or "staff_base" in col.lower() or "facilities_base" in col.lower() or "value_for_money_base" in col.lower()]
X = X.drop(columns=leakage_cols)

X["location_base"] = X["score_location"] / (X["location_base"] + 1e-6)

y = df["country_group"]
le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train = X_train.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
X_test  = X_test.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)




model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)
model.fit(X_train, y_train)


importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values(by="importance", ascending=False)

print(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(importance_df["feature"], importance_df["importance"], color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()


y_train_pred = model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f" Training Accuracy: {train_acc:.4f}")
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
plt.figure(figsize=(5, 5))
plt.bar(["Training", "Test"], [train_acc, test_acc], color=["skyblue", "lightcoral"])
plt.title("Training vs Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
print(" Accuracy:", accuracy_score(y_test, y_pred))
print("\n Classification Report:")
print(classification_report(y_test, y_pred))



report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()

class_report_df = report_df.iloc[:-3, :]


plt.figure(figsize=(10, 6))
sns.barplot(data=class_report_df[['precision', 'recall', 'f1-score']])
plt.title("Classification Metrics per Class")
plt.xlabel("Metric")
plt.ylabel("Score (0–1)")
plt.ylim(0, 1)
plt.legend(labels=['Precision', 'Recall', 'F1-Score'])
plt.tight_layout()
plt.show()
print("\n Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Dropped age and country because they are the lowest in feature importance



In [ ]:
df = pd.read_csv("dataset/final_dataset.csv")

X = df.drop(columns=["country_group"], errors='ignore')
dropped_cols = [col for col in X.columns if "cleanliness_base" in col.lower() or  "facilities_base" in col.lower() or "value_for_money_base" in col.lower() or "age_group" in col.lower() or "country" in col.lower()]
X = X.drop(columns=dropped_cols)



X["location_base"] = X["score_location"] / (X["location_base"] )
X["comfort_base"] = X["score_comfort"] / (X["comfort_base"] )
X["staff_base"] = X["score_staff"] / (X["staff_base"] )
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train = X_train.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
X_test  = X_test.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)
model.fit(X_train, y_train)
importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values(by="importance", ascending=False)

print(importance_df)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(importance_df["feature"], importance_df["importance"], color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

y_train_pred = model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f"🎯 Training Accuracy: {train_acc:.4f}")
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
plt.figure(figsize=(5, 5))
plt.bar(["Training", "Test"], [train_acc, test_acc], color=["skyblue", "lightcoral"])
plt.title("Training vs Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred))
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import pandas as pd


report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()


class_report_df = report_df.iloc[:-3, :]


plt.figure(figsize=(10, 6))
sns.barplot(data=class_report_df[['precision', 'recall', 'f1-score']])
plt.title("Classification Metrics per Class")
plt.xlabel("Metric")
plt.ylabel("Score (0–1)")
plt.ylim(0, 1)
plt.legend(labels=['Precision', 'Recall', 'F1-Score'])
plt.tight_layout()
plt.show()
print("\n🧩 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Use all base all scores all categorical

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder


df = pd.read_csv("dataset/final_dataset.csv")


df = df.dropna(subset=["country_group"])


X = df.drop(columns=["country_group"], errors='ignore')


X["location_base"] = X["score_location"] / (X["location_base"])
X["comfort_base"] = X["score_comfort"] / (X["comfort_base"] )
X["staff_base"] = X["score_staff"] / (X["staff_base"] )
X["cleanliness_base"] = X["score_cleanliness"] / (X["cleanliness_base"] )
X["facilities_base"] = X["score_facilities"] / (X["facilities_base"] )
X["value_for_money_base"] = X["score_value_for_money"] / (X["value_for_money_base"])
y = df["country_group"]
le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train = X_train.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
X_test  = X_test.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)



from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)
model.fit(X_train, y_train)


importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values(by="importance", ascending=False)

print(importance_df)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(importance_df["feature"], importance_df["importance"], color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

y_train_pred = model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f" Training Accuracy: {train_acc:.4f}")
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
plt.figure(figsize=(5, 5))
plt.bar(["Training", "Test"], [train_acc, test_acc], color=["skyblue", "lightcoral"])
plt.title("Training vs Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)

for i, acc in enumerate([train_acc, test_acc]):
    plt.text(i, acc + 0.02, f"{acc:.3f}", ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()
print(" Accuracy:", accuracy_score(y_test, y_pred))
print("\n Classification Report:")
print(classification_report(y_test, y_pred))
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import pandas as pd


report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()


class_report_df = report_df.iloc[:-3, :]


plt.figure(figsize=(10, 6))
sns.barplot(data=class_report_df[['precision', 'recall', 'f1-score']])
plt.title("Classification Metrics per Class")
plt.xlabel("Metric")
plt.ylabel("Score (0–1)")
plt.ylim(0, 1)
plt.legend(labels=['Precision', 'Recall', 'F1-Score'])
plt.tight_layout()
plt.show()

print("\n Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Final Model All score all base user gender, Age range and traveller type

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder


df = pd.read_csv("dataset/final_dataset.csv")


df = df.dropna(subset=["country_group"])


X = df.drop(columns=["country_group"], errors='ignore')

X = X.loc[:, ~X.columns.str.startswith("user_country_")]


X["location_base"] = X["score_location"] / (X["location_base"] )
X["comfort_base"] = X["score_comfort"] / (X["comfort_base"] )
X["staff_base"] = X["score_staff"] / (X["staff_base"] )
X["cleanliness_base"] = X["score_cleanliness"] / (X["cleanliness_base"] )
X["facilities_base"] = X["score_facilities"] / (X["facilities_base"] )
X["value_for_money_base"] = X["score_value_for_money"] / (X["value_for_money_base"] )
y = df["country_group"]
le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train = X_train.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
X_test  = X_test.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)



from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)
model.fit(X_train, y_train)


importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values(by="importance", ascending=False)

print(importance_df)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(importance_df["feature"], importance_df["importance"], color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

y_train_pred = model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
print(f" Training Accuracy: {train_acc:.4f}")
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
plt.figure(figsize=(5, 5))
plt.bar(["Training", "Test"], [train_acc, test_acc], color=["skyblue", "lightcoral"])
plt.title("Training vs Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)

for i, acc in enumerate([train_acc, test_acc]):
    plt.text(i, acc + 0.02, f"{acc:.3f}", ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()
print(" Accuracy:", accuracy_score(y_test, y_pred))
print("\n Classification Report:")
print(classification_report(y_test, y_pred))
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import pandas as pd


report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()


class_report_df = report_df.iloc[:-3, :]


plt.figure(figsize=(10, 6))
sns.barplot(data=class_report_df[['precision', 'recall', 'f1-score']])
plt.title("Classification Metrics per Class")
plt.xlabel("Metric")
plt.ylabel("Score (0–1)")
plt.ylim(0, 1)
plt.legend(labels=['Precision', 'Recall', 'F1-Score'])
plt.tight_layout()
plt.show()

print("\n Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
import shap
import numpy as np
import pandas as pd


X_train = pd.DataFrame(X_train, columns=X.columns)
X_test  = pd.DataFrame(X_test, columns=X.columns)


background = shap.sample(X_train, 100, random_state=42)
X_test_sample = shap.sample(X_test, 300, random_state=42)


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_sample)
print(" SHAP values shape:", np.array(shap_values).shape)


In [ ]:
print("Type of shap_values:", type(shap_values))
if isinstance(shap_values, list):
    for i, arr in enumerate(shap_values):
        print(f"Class {i}: {arr.shape}")
else:
    print("shap_values shape:", shap_values.shape)

print("X_test_sample shape:", X_test_sample.shape)

In [ ]:
import numpy as np


shap_values_mean = np.mean(np.abs(shap_values), axis=2)

shap.summary_plot(
    shap_values_mean,
    X_test_sample,
    feature_names=X_train.columns,
    plot_type="bar"
)

In [ ]:
i = 211
pred_class = model.predict(X_test_sample.iloc[[i]])[0]
class_ix = pred_class


shap.initjs()
shap.force_plot(
    explainer.expected_value[class_ix],
    shap_values[i, :, class_ix],
    X_test_sample.iloc[i],
    feature_names=X_train.columns
)

In [ ]:
print(df.groupby("country_group")[["score_overall","score_location","score_staff"]].mean())


In [ ]:
# @title
%pip install lime


In [ ]:
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd


In [ ]:
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test  = pd.DataFrame(X_test, columns=X.columns)


In [ ]:
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=le.classes_.tolist(),
    discretize_continuous=True,
    mode='classification'
)


In [ ]:
j = 2

exp = explainer_lime.explain_instance(
    X_test.iloc[j].values,
    model.predict_proba,
    num_features=10,
    top_labels=len(le.classes_)
)


exp.show_in_notebook(show_table=True)
